In [93]:
import stim
import numpy as np
import pandas as pd

In [ ]:
def soft_wrapper(detector_file, logic_file, output_file, alpha=1.0, log_base='nat', eps=1e-12):
    # Load data
    det_df = pd.read_csv(detector_file, header=None)
    # num_detectors = det_df.shape[1]
    # det_df.columns = [f"Det{i+1}" for i in range(num_detectors)]
    logic_df = pd.read_csv(logic_file, header=None)
    logic_df.columns = ["LogicBit"]

    # Compute soft detector features
    det_df["DetMean"] = det_df.mean(axis=1)
    det_df["DetVar"] = det_df.var(axis=1)

    # Compute soft logical bit probability
    # Example: logistic function of detector mean
    det_df["LogicProb"] = 1 / (1 + np.exp(-alpha * det_df["DetMean"]))

    # Logic entropy from LogicProb
    p = np.clip(det_df['LogicProb'].values, eps, 1 - eps)
    H_logic = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    if log_base == 'bit':
        H_logic = H_logic / np.log(2)
    det_df['LogicEntropy'] = H_logic

    # Detector entropy from DetVar (rough univariate proxy)
    detvar = np.maximum(det_df['DetVar'].values, eps)
    H_det = 0.5 * np.log(2 * np.pi * np.e * detvar)
    if log_base == 'bit':
        H_det = H_det / np.log(2)
    det_df['DetectorEntropy'] = H_det

    # Optionally replace hard bit with probability
    merged = pd.concat([det_df, logic_df], axis=1)

    # Save unified dataset
    merged.to_csv(output_file, index=False)

In [ ]:
def gen_dem(distance=3, rounds=5, p=1e-3, shots=20000):
    circ = stim.Circuit.generated("surface_code:rotated_memory_z",
                                  distance=distance, 
                                  rounds=rounds, 
                                  after_clifford_depolarization=p, # Apply depolarizing noise after Clifford gates (idk what those are atm)
                                  before_round_data_depolarization=p, # Apply depolarizing noise before each round
                                  before_measure_flip_probability=p, # Measurement errors
                                  after_reset_flip_probability=p # Reset errors
                                  )
    
    # get detector and observable samples
    sampler = circ.compile_detector_sampler()
    det_samples, obs_samples = sampler.sample(shots=shots, separate_observables=True)
    # det_samples = sampler.sample(shots=shots, separate_observables=False)

    
    # get analog readout wrapper
    measurement_sampler = circ.compile_sampler()
    m2d = circ.compile_m2d_converter()
    measurements = measurement_sampler.sample(shots)
    detectors, observables = m2d.convert(measurements=measurements, separate_observables=True)
    
    return det_samples, obs_samples, measurements, detectors, observables

In [95]:
det_samples, logical_labels, measurement_samples, detectors, observables = gen_dem(distance=3, rounds=5, p=1e-3, shots=20000)

print(f"Detector samples shape: {det_samples.shape}")
print(f"Logical observable sample shape: {logical_labels.shape}")
print(f"Measurement samples shape: {measurement_samples.shape}")
print(f"Detectors shape: {detectors.shape}")
print(f"Observables shape: {observables.shape}")

Detector samples shape: (20000, 40)
Logical observable sample shape: (20000, 1)
Measurement samples shape: (20000, 49)
Detectors shape: (20000, 40)
Observables shape: (20000, 1)


In [78]:
np.savez_compressed('../data/surface_code_data.npz',
                    detectors=det_samples,
                    observables=logical_labels)

In [86]:
np.savetxt('../data/detector_samples.csv', det_samples, delimiter=',', fmt='%d')
np.savetxt('../data/logical_labels.csv', logical_labels, delimiter=',', fmt='%d')
np.savetxt('../data/observables.csv', observables, delimiter=',', fmt='%d')

In [92]:
soft_wrapper("../data/detector_samples.csv", "../data/logical_labels.csv", "../data/detector_soft_data.csv", alpha=3.0)